In [2]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath("../src"))

In [3]:
df = pd.read_csv('../data/labeledTrainData.tsv.zip', header=0, delimiter="\t", quoting=3)
df.describe()

,sentiment
count,25000.00000
mean,0.50000
std,0.50001
min,0.00000
25%,0.00000
50%,0.50000
75%,1.00000
max,1.00000


In [4]:
import my_utils # noqa: F401

wordlist = df.review.nlp.review_to_wordlist()
wordlist.shape

(25000,)

### TF‑IDF representation  

Convert the corpus into a sparse TF‑IDF matrix.

max_df=0.85 removes extremely common words.

English stopwords are removed automatically.

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english', max_df=0.85)
tfidf_matrix = vectorizer.fit_transform(df.review)
tfidf_matrix.shape

(25000, 74538)

In [41]:
from sklearn.decomposition import TruncatedSVD

n_components=200

svd = TruncatedSVD(n_components=n_components, random_state=42)
X_reduced = svd.fit_transform(tfidf_matrix)
X_reduced.shape

(25000, 200)

### Baseline sentiment models  

Split the reduced TF‑IDF features into train/test sets to evaluate simple classifiers.

This establishes a baseline before experimenting with more advanced embeddings.

In [42]:
from sklearn.model_selection import train_test_split


X_train_sentiment, X_test_sentiment, y_train_sentiment, y_test_sentiment = train_test_split(
    X_reduced, df.sentiment, test_size=0.2, random_state=42
)

### Baseline models  

Evaluate several linear and tree‑based classifiers on the reduced TF‑IDF features.

Logistic Regression and Linear SVM are strong baselines for text classification.

In [43]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

evaluators = [
    (LogisticRegression(max_iter=500), 'LogisticRegression'),
    (LinearSVC(), 'LinearSVC'),
    (RandomForestClassifier(random_state=42, max_depth=10), 'RandomForestClassifier')    
]


In [40]:
for tup in evaluators:
    (clf, eval_class) = tup
    
    print(f'Evaluating {eval_class}')
    clf.fit(X_train_sentiment, y_train_sentiment)
    score = clf.score(X_test_sentiment, y_test_sentiment)    
    print(f'{eval_class} sentiment score: {score}')
    print("\n")

Evaluating LogisticRegression
LogisticRegression sentiment score: 0.8702


Evaluating LinearSVC
LinearSVC sentiment score: 0.8762


Evaluating RandomForestClassifier
RandomForestClassifier sentiment score: 0.8026




### Summary  

Baseline classifiers trained on reduced TF‑IDF features provide a reference point for future iterations.

Next steps include experimenting with Word2Vec embeddings and improving preprocessing.